In [1]:
import random
import secrets
from typing import Literal
import pandas as pd


class MontyHallSimulation:
    def __init__(self):
        self.__doors: list[Literal["0", "1", "0X", "1X", "-"]] = ["0", "0", "0"]
        self.reward_position: int | None = None
        self.selected_door: int | None = None

    def _set_reward(self):
        _ = int(secrets.choice("012"))
        self.reward_position = _
        self.__doors[_] = "1"

    def _choose_door(self, n: int):
        self.selected_door = n
        self.__doors[n] = self.__doors[n] + "X"

    def _open_door_with_no_reward(self):
        while True:
            choosen_idx = int(secrets.choice("012"))
            # print("Doors ", self.__doors)
            # print("Choosen: ", choosen_idx)
            # print()
            if self.__doors[choosen_idx] == "0" and self.__doors[choosen_idx] not in ["0X", "1X"]:
                break
        self.__doors[choosen_idx] = "-"
        # print(self.__doors)

    def _change_answer(self):
        list_of_idx_that_is_not_opened_and_marked = []
        for i in range(len(self.__doors)):
            if self.__doors[i] not in ["0X", "1X", "-"]:
                list_of_idx_that_is_not_opened_and_marked.append(i)

        random.shuffle(list_of_idx_that_is_not_opened_and_marked)

        self.__doors[self.selected_door] = self.__doors[self.selected_door].replace("X", "")
        self.__doors[list_of_idx_that_is_not_opened_and_marked[0]] = self.__doors[list_of_idx_that_is_not_opened_and_marked[0]] + "X"

    def restart(self):
        self.__doors = ["0", "0", "0"]
        self.selected_door = None

    def start(self, i: int, change_ans: bool | float) -> bool:
        self._set_reward()
        self._choose_door(n=i)
        self._open_door_with_no_reward()

        if isinstance(change_ans, float):
            change_ans = bool(random.binomialvariate(n=1, p=change_ans))

        if isinstance(change_ans, bool) and not change_ans:
            return ("1X" in self.__doors) == True

        self._change_answer()
        return ("1X" in self.__doors) == True


In [2]:
def simulate(n: int, change_ans: bool|float) -> float:
    result = []
    for _ in range(n):
        simulator = MontyHallSimulation()
        is_win = simulator.start(int(secrets.choice("012")), change_ans=change_ans)
        result.append(is_win)

    df_no_change_ans = pd.DataFrame(result, columns=["Win"])

    return df_no_change_ans.loc[df_no_change_ans["Win"]].shape[0]/df_no_change_ans.shape[0]


In [3]:
n = 200_000
stay = simulate(n=n, change_ans=False)
change_ans = simulate(n=n, change_ans=True)
stay_50_change_ans_50 = simulate(n=n, change_ans=0.5)

print(f"Stay              : {round(stay*100, 2)}%")
print(f"Change ans        : {round(change_ans*100, 2)}%")
print(f"50-50 stay change : {round(stay_50_change_ans_50*100, 2)}%")


Stay              : 33.46%
Change ans        : 66.71%
50-50 stay change : 50.12%
